# Object Detection

Apply your new knowledge of CNNs to one of the hottest (and most challenging!) fields in computer vision: object detection.

**Learning Objectives**
* Identify the components used for object detection (landmark, anchor, bounding box, grid, ...) and their purpose
* Implement object detection
* Implement non-max suppression to increase accuracy
* Implement intersection over union
* Handle bounding boxes, a type of image annotation popular in deep learning
* Apply sparse categorical crossentropy for pixelwise prediction
* Implement semantic image segmentation on the CARLA self-driving car dataset
* Explain the difference between a regular CNN and a U-net
* Build a U-Net

---

## Table of Contents

- [Object Detection](#object-detection)
- [Object Localization](#object-localization)
- [Landmark Detection](#landmark-detection)
- [Sliding Windows Detection](#object-detection)
- [Convolutional Implementation of Sliding Windows](#convolutional-implementation-of-sliding-windows)
- [Bounding Box Predictions](#bounding-box-predictions)
- [Intersection Over Union](#intersection-over-union)
- [Non-Max Suppression](#non-max-suppression)
- [Anchor Boxes](#anchor-boxes)
- [YOLO Algorithm](#yolo-algorithm)
- [Region Proposal](#region-proposal)
- [Semantic Segmentation with U-Net](#semantic-segmentation-with-u-net)
- [Transpose Convolutions](#transpose-convolutions)
- [U-Net Architecture Intuition](#u-net-architecture-intuition)
- [U-Net Architecture](#u-net-architecture)

---

## Object Localization

This section introduces the transition from simple **Image Classification** to **Object Localization** and eventually **Object Detection**. Here is the summary of the key concepts.

* **Classification:** Identifying *what* is in an image (e.g., "this is a car").
* **Classification with Localization:** Identifying the object *and* drawing a bounding box around it (usually for a single object).
* **Detection:** Identifying and localizing *multiple* objects of different categories within the same image.

<div style='display:flex; justify-content:center'>
    <img src='images/class_local_detection.png' width='750px'>
</div>

### Defining the Bounding Box

To localize an object, the network is trained to output four specific real numbers:

* **($b_x, b_y$):** The coordinates of the center point of the object.
* **$b_h$:** The height of the bounding box (expressed as a fraction of total image height).
* **$b_w$:** The width of the bounding box (expressed as a fraction of total image width).
* **Coordinate System:** The top-left of the image is $(0, 0)$ and the bottom-right is $(1, 1)$.

<div style='display:flex; justify-content:center'>
    <img src='images/localization.png' width='750px'>
</div>

### The Target Label Vector ($y$)

In a supervised learning task for localization, the target label $y$ is structured as a multi-component vector (in this example, an 8-element vector):

$$y = [p_c, b_x, b_y, b_h, b_w, c_1, c_2, c_3]^T$$

* **$p_c$ (Probability of Class):** A binary flag where 1 means an object is present and 0 means only background is present.
* **$b_x, b_y, b_h, b_w$:** The bounding box parameters.
* **$c_1, c_2, c_3$:** The class labels (e.g., Pedestrian, Car, Motorcycle).
* **"Don't Cares":** If $p_c=0$, all other elements in the vector are ignored during training.

### The Loss Function

The loss function is calculated differently depending on whether an object is present in the ground truth:

* **If Object Present ($y_1 = 1$):** The loss is typically the sum of squared errors across all 8 components:

$$L(\hat y, y) = \Sigma_{i=1}^8 (\hat y_i - y_i)^2$$

* **If No Object ($y_1 = 0$):** The loss only considers the error in the $p_c$ prediction:

$$L(\hat y, y) = (\hat y_i - y_i)^2$$

> **Note:** While squared error is used for simplicity, specialized losses like **Log Loss** for classes ($c_i$) and $p_c$, or **IoU-based losses** for bounding boxes, are often used in advanced applications. In the subsequent sections, we will cover these loss functions.

---

## Landmark Detection

This section explains **Landmark Detection**, a more granular version of localization where the neural network predicts specific coordinates for points of interest rather than just a general bounding box.

### Key Concepts in Landmark Detection

* **Beyond Bounding Boxes:** Instead of just outputting width and height, the network outputs the specific ($x, y$) coordinates for multiple "landmarks" (key points) that define the shape or pose of an object.
* **Vector Construction:** To detect $N$ landmarks, the network outputs a vector $y$ containing:
    * $P_c$: A confidence score (is an object present?).
    * $2N$ coordinates: $(l_{1x}​,l_{1y}​,l_{2x}​,l_{2y}​,\dots, l_{Nx}​,l_{Ny}​)$.
    * *Example:* For 64 facial landmarks, the model would have 129 output units.
* **Consistency is Critical:** For the model to learn, the "identity" of each landmark must be identical across every image in the dataset (e.g., $l_1$ must *always* represent the left corner of the left eye).

<div style='display:flex; justify-content:center'>
    <img src='images/landmark_detection.png' width='750px'>
</div>

### Common Applications

* **Facial Expression & Emotion Recognition:** By tracking landmarks around the mouth and eyes, models can determine if a person is smiling, frowning, or surprised.
* **Augmented Reality (AR) Filters:** Apps like Snapchat use these landmarks to warp faces or accurately place digital objects (like crowns or hats) on a user's head.
* **Human Pose Estimation:** Detecting the "skeleton" of a person by placing landmarks on joints (shoulders, elbows, wrists, knees) to recognize activities or body language.

### Implementation Summary

| Feature | Bounding Box (Localization) | Landmark Detection |
| --- | --- | --- |
| **Output Type** | 4 Numbers ($b_x​,b_y​,b_h​,b_w$) | 2 $\times$ Numbers (Coordinates) |
| **Detail Level** | General position | Specific shape/contour/pose |
| **Data Effort** | Low (Box coordinates) | High (Laborious point annotation) |
| **Model Type** | Regression task | Regression task |

---

## Object Detection

This section introduces the **Sliding Windows Detection Algorithm**, the foundational (though computationally expensive) method for transitioning from simple image classification to full object detection.

### Key Concepts of Sliding Windows

* **Training Phase:**
    * The process begins by training a standard Convolutional Neural Network (ConvNet) on a dataset of *closely cropped* images.
    * The training images ($x$) are cropped so that the object (e.g., a car) is centered and fills most of the frame.
    * The model learns a binary classification ($y$): **1** if a car is present, **0** if not.
* **Detection Phase (The "Sliding" Process):**
    * **Window Selection:** A small rectangular window is chosen and placed over the test image.
    * **Cropping & Classifying:** The region within the window is cropped out and fed into the pre-trained ConvNet to check for the object.
    * **Iteration (Stride):** The window is shifted (using a specific stride) across the entire image, repeating the classification at every step.
    * **Multi-Scale Detection:** To find objects of different sizes, the entire process is repeated multiple times using progressively larger window sizes.

### The Computational Challenge

While conceptually simple, this algorithm has a significant drawback: **High Computational Cost.**

* **The Dilemma:**
    * If you use a *large stride*, you run fewer windows (saving time) but might miss the object or localize it poorly.
    * If you use a *small stride*, you get better accuracy but must run the ConvNet hundreds or thousands of times for a single image.
* **Historical Context:** In the past, sliding windows worked well because classifiers were "cheap" (simple linear functions). Modern ConvNets are "expensive" (deep architectures), making the traditional independent sliding window approach inefficiently slow.

### Summary of the Pipeline

| Stage | Action | Result |
| --- | --- | --- |
| **1. Dataset Prep** | Crop images tightly around the object. | Specialized "Car vs. Not Car" classifier. |
| **2. Initial Pass** | Slide a small window with a fixed stride. | Detects small or distant objects. |
| **3. Scaling** | Increase window size and repeat the slide. | Detects medium and large objects. |
| **4. Classification** | Run each crop through the ConvNet. | Generate a map of object detections. |

---

## Convolutional Implementation of Sliding Windows

This section explains a major breakthrough in object detection efficiency: *Convolutional Implementation of Sliding Windows*. This method allows you to evaluate multiple image regions simultaneously in a single forward pass, rather than cropping and running a network hundreds of times.

### Turning Fully Connected (FC) Layers into Convolutional Layers

To run sliding windows convolutionally, you must first convert the "head" of your network from FC layers to convolutional layers.

* **The Conversion Logic:** A fully connected layer with 400 neurons, for example, can be replaced by a convolutional layer of $1 \times 1 \times 400$ volume using 400 $5 \times 5 \times depth$ filters (assuming the input volume is $5 \times 5 \times depth$). Mathematically, these are identical because each filter "looks" at the entire previous volume, just as a neuron is connected to all previous activations.
* **1x1 Convolutions:** Subsequent FC layers are replaced by $1 \times 1$ convolutional layers, maintaining the $1 \times 1 \times depth$ volume until the final output layer (e.g., $1 \times 1 \times 4$ for 4 classes).

<div style='display:flex; justify-content: center'>
    <img src='images/sliding_win_convnet2.png' width=850px>
</div>

### Convolutional Sliding Windows (The OverFeat Approach)

Once the network is fully convolutional, we can input an image *larger* than the original training size.

* **Shared Computation:** Instead of sliding a window and re-calculating the entire network for every crop, the convolutional layers share computations for overlapping pixels.
* **Output Volumes:** If we train on $14 \times 14$ images and then input a $16 \times 16$ image, the network will naturally output a $2 \times 2 \times 4$ volume instead of a single $1 \times 1 \times 4$ prediction.
* **Spatial Mapping:** Each "pixel" in that $2 \times 2$ output volume corresponds to a specific $14 \times 14$ window in the original image.
    * *Top-Left Pixel:* Result of the top-left $14 \times 14$ window.
    * *Bottom-Right Pixel:* Result of the bottom-right $14 \times 14$ window.

<div style='display:flex; justify-content: center'>
    <img src='images/sliding_win_convnet1.png' width=850px>
</div>


### Efficiency Summary

| | Traditional Sliding Windows | Convolutional Implementation |
| --- | --- | --- |
| **Method** | Crop, resize, and run independently. | Run entire image in one forward pass. |
| **Computation** | Massive duplication of math. | Shares math for overlapping regions. |
| **Speed** | Infeasibly slow for deep ConvNets. | Extremely fast and efficient. |
| **Output** | A single classification per run. | A "heat map" of classifications. |

### Key Takeaways

* **The Stride:** The stride of your sliding window is determined by the pooling layers in your network. For instance, a  max pool effectively creates a stride of 2 in the original image.
* **One Pass:** This allows you to process an entire  image and get an  grid of predictions in the same time it would take to do a few independent crops.
* **The Remaining Problem:** While fast, this method still relies on fixed grids, meaning the bounding boxes may not align perfectly with the object.

---

## Bounding Box Predictions

This section introduces **YOLO (You Only Look Once)**, a revolutionary object detection algorithm that provides high accuracy and real-time performance by treating detection as a single regression problem.

### The YOLO Core Concept

* **Grid Division:** The input image (e.g., $100 \times 100$) is divided into an $S \times S$ grid (e.g., $3 \times 3$ for illustration, usually $19 \times 19$ in practice).
* **The Midpoint Rule:** Each object in the image is assigned to *exactly one* grid cell—specifically, the one containing the object’s **midpoint**.
* **Single Forward Pass:** Unlike sliding windows, YOLO uses a single Convolutional Neural Network pass to predict the entire output volume at once.

<div style='display: flex; justify-content: center'>
    <img src='images/yolo_algorithm.png' width='750px'>
</div>

### The Output Volume ($y$)

The network doesn't just output a class; it outputs a 3D volume representing the grid and its predictions.

* **Vector Structure:** For each cell, the target vector  typically contains 8 components:
    * $P_c$: Confidence score (1 if an object is present, 0 if background).
    * $b_x, b_y$: Midpoint coordinates relative to the specific grid cell.
    * $b_h, b_w$: Height and width of the bounding box relative to the grid cell.
    * $c_1, c_2, c_3$: One-hot encoded class probabilities (e.g., Pedestrian, Car, Motorcycle).

* **Volume Dimensions:** For a $3 \times 3$ grid with 8 parameters per cell, the output is a $3 \times 3 \times 8$ tensor.

### Bounding Box Parameterization

YOLO uses a clever relative coordinate system to keep numbers manageable:

* **Local Coordinates:** Within a grid cell, the top-left is $(0, 0)$ and the bottom-right is ($1, 1$).
* **$b_x, b_y$:** Must be between 0 and 1 because the midpoint is strictly inside that cell.
* **$b_w, b_h$:** Can be greater than 1 if the object is larger than the grid cell (e.g., a car spanning three cells).

### Why YOLO is a "Gold Standard"

* **Accuracy:** Because it outputs $b_x​,b_y​,b_h​,b_w$ as continuous numbers, it can predict boxes of any aspect ratio and size, far exceeding the precision of a fixed-stride sliding window.
* **Speed:** It is a convolutional implementation with massive shared computation. This efficiency makes it capable of "Real-Time" detection (processing video frames at 30+ FPS).
* **Global Context:** Because the network sees the entire image at once, it encodes contextual information about classes and their appearance.

### Summary Comparison

| Feature | Sliding Windows (Conv) | YOLO |
| --- | --- | --- |
| **Localization** | Restricted by stride/grid. | Precise (outputs real numbers). |
| **Output Type** | Class Heatmap. | Full bounding box regression. |
| **Aspect Ratios** | Usually fixed squares. | Flexible (rectangles of any size). |
| **Complexity** | Simple, but less accurate. | Complex to train, but very fast. |

---

## Intersection Over Union

This section explains the **Intersection over Union (IoU)** function, which is the primary metric used to evaluate the accuracy of object detection algorithms.

### Key Concepts of IoU

* **Definition:** IoU is a numerical value that measures the overlap between two bounding boxes: the **Ground Truth** (the actual location) and the **Predicted Box** (the algorithm's output).
* **The Formula:** It is calculated by dividing the area of overlap by the total combined area of both boxes.

$$IoU = \frac{\text{Size of Intersection}}{\text{Size of Union}}$$

* **The Scale:** 
    * **0**: No overlap at all.
    * **1**: Perfect overlap (Intersection equals Union).
* **The "Correctness" Threshold:** By convention, a prediction is typically considered "correct" if the $IoU \ge 0.5$.
    * This is a human-chosen convention; more stringent tasks might require 0.6 or 0.7.
    * Thresholds below 0.5 are rarely used in professional benchmarks.

### Dual Purposes of IoU

The section highlights that IoU isn't just for a final grade; it's a functional tool:

1. **Evaluation:** It provides a way to turn spatial coordinates into an "Accuracy" score. You can count how many objects were "correctly" localized based on the IoU threshold.
2. **Algorithm Optimization:** It is used as a core component in **Non-Max Suppression (NMS)**, which helps clean up multiple detections of the same object (detailed in the next video).

### Summary Comparison

| Feature | Low IoU ($\lt 0.5$) | High IoU ($\ge 0.5$) |
| --- | --- | --- |
| **Visual Match** | Poor overlap; box is off-center or wrong size. | Decent to excellent overlap. |
| **Status** | False Positive / Incorrect Localization. | True Positive / Correct Localization. |
| **Mathematical Meaning** | Large Union, tiny Intersection. | Intersection area is at least half of the Union. |

---

## Non-Max Suppression

This section explains **Non-Max Suppression (NMS)**, a critical post-processing step in object detection used to ensure that each object is detected only once, even if multiple grid cells "raise their hand" to claim it.

### The Core Problem: Multiple Detections

In algorithms like YOLO, the same object (e.g., a car) often spans several grid cells. Even though only one cell is technically responsible for the object's midpoint, in practice:

* Multiple cells may predict high object detection probabilities ($P_c$).
* This results in several overlapping bounding boxes for a single physical object.
* NMS "cleans up" these redundant boxes to leave one final, most confident detection.

### The Non-Max Suppression Algorithm

The algorithm follows a "greedy" approach to filter detections:

1. **Discard Low-Probability Boxes:** First, eliminate all boxes with a confidence score ($P_c$) below a certain threshold (e.g., $P_c \le 0.6$).
2. **Pick the "Winner":** From the remaining boxes, select the one with the *highest* probability. This is committed as a final prediction.
3. **Suppress Overlaps:** Look at all other remaining boxes. If a box has a high **Intersection over Union (IoU)** with the "Winner" (e.g., $IoU \ge 0.5$ ), it is discarded (suppressed).
4. **Repeat:** Go back to step 2 and pick the next highest remaining probability until all boxes have been either selected or suppressed.

<div style='display:flex; justify-content: center'>
    <img src='images/non_max_supression.png' width='600px'>
</div>

### Key Components and Parameters

* **$P_c$ (Probability Score):** Typically calculated as $P_c \times \text{CLass Probability}$. It tells the model which box is the most reliable "starting point."
* **IoU Threshold:** A hyperparameter that defines how much overlap is "too much."
    * If set to **0.5**, any box overlapping the winner by 50% or more is removed.
* **Class Independence:** If detecting multiple classes (e.g., cars and pedestrians), NMS is typically run independently for each class.

### Key Takeaway

"Non-Max" simply means we keep the **Maximal** probability box and **Suppress** the ones that are not the maximum. This makes YOLO incredibly fast because it's a simple post-processing step that doesn't require further neural network passes.

---

## Anchor Boxes

This section explains the concept of **Anchor Boxes**, a method that allows the YOLO algorithm to detect multiple objects within a single grid cell and improve the specialization of the neural network.

### The Core Problem: Overlapping Midpoints

In the basic YOLO model, each grid cell can only detect one object. If two objects (e.g., a pedestrian standing in front of a car) have midpoints that fall into the same grid cell, the model would normally be forced to choose only one to detect.

### How Anchor Boxes Work

Instead of assigning one object per cell, you pre-define a set of shapes called **Anchor Boxes**.

* **Assignment:** Each object in the training set is assigned to a specific (Grid Cell, Anchor Box) pair.
* **Selection Criteria:** The object is assigned to the anchor box with which its bounding box has the highest IoU.
* **Specialization:** Anchor boxes allow the model to specialize. For example, Anchor Box 1 might specialize in "tall, skinny" objects (pedestrians), while Anchor Box 2 specializes in "wide, horizontal" objects (cars).

### Changes to the Target Label ($Y$)

Using anchor boxes changes the structure of the output vector. If you have 3 classes and 2 anchor boxes, the vector for each grid cell doubles from 8 elements to 16:

* $Y = [\text{Anchor Box 1 parameters}, \text{Anchor Box 2 parameters}]^T$
* **Output Volume:** For a $3 \times 3$ grid with 2 anchor boxes and 8 parameters each ($P_c​,b_x​,b_y​,b_h​,b_w​,c1​,c2​,c3$​), the output volume becomes $3 \times 3 \times 16$ (or $3 \times 3 \times 2 \times 8$).

<div style='display:flex; justify-content: center'>
    <img src='images/anchor_box_examople.png' width='750px'>
</div>

### Choosing Anchor Box Shapes

* **Manual Selection:** Historically, researchers chose 5 to 10 shapes by hand that covered common object types (e.g., squares, tall rectangles, wide rectangles).
* **Automatic Selection (K-means):** More advanced versions of YOLO (like YOLOv3) use **K-means clustering** on the training dataset to automatically find the most representative bounding box shapes for the specific data.

### Key Takeaway

While anchor boxes solve the "multiple objects in one cell" problem, this scenario is actually rare in a fine $19 \times 19$ grid. The primary benefit of anchor boxes in modern systems is actually the **specialization** they provide, making it easier for the network to learn distinct features for vastly different object shapes.

---

## YOLO Algorithm

This section synthesizes the individual components (Grid cells, Anchor Boxes, and IoU) into the complete **YOLO** algorithm, covering the full pipeline from training set construction to final prediction.

### Putting it All Together: The YOLO Pipeline

* **Training Set Construction:**
    * The image is divided into a grid (e.g., $3 \times 3$ or $19 \times 19$).
    * For each grid cell, a target vector $Y$ is created. If using 2 anchor boxes and 3 classes, the vector is 16-dimensional ($2 \times (1 + 4 + 3)$).
    * **Assignment:** An object is assigned to a specific grid cell (based on its midpoint) and a specific anchor box (based on the highest IoU with the anchor's shape).
    * **Empty Cells:** If a cell contains no object midpoint, $P_c$ for all anchors is set to 0, and the remaining parameters are treated as "don't cares."
* **Model Architecture:**
    * The model is a standard ConvNet that takes a full image as input.
    * The output is a single volume (e.g., $3 \times 3 \times 16$). The network makes predictions for every grid cell and every anchor box simultaneously in one forward pass.
* **Making Predictions:**
    * At test time, the model outputs the full tensor.
    * For cells without objects, the network will still output numbers (noise), but these are ignored because the predicted  (probability of an object) will be low.
    * For cells with objects, the network provides the specific $b_x​,b_y​,b_h​,b_w​$ coordinates and class probabilities for the anchor box that best fits the object.
* **Post-Processing (Non-Max Suppression):**
    * Because many grid cells might predict the same object, the raw output often contains multiple overlapping boxes.
    * **Step 1:** Discard boxes with low $P_c$.
    * **Step 2:** For each class (Pedestrian, Car, Motorcycle), run *Non-Max Suppression* independently to filter out redundant detections and keep only the most confident, accurate bounding box.

### Summary of Output Dimensions

The shape of the output volume is determined by the grid and the number of anchors:

$$Volume = S \times S \times (N \times (5 + C))$$

* **$S$:** Grid size (e.g., 19).
* **$N$:** Number of anchor boxes.
* **5:** Parameters for $P_c​,b_x​,b_y​,b_h​,b_w​$.
* **$C$:** Number of classes.

### Key Takeaway

* YOLO is a state-of-the-art object detection model that is fast and accurate
* It runs an input image through a CNN, which outputs a $19 \times 19 \times 5 \times 85$ dimensional volume. 
* The encoding can be seen as a grid where each of the 19x19 cells contains information about 5 boxes.
* You filter through all the boxes using non-max suppression. Specifically: 
    * Score thresholding on the probability of detecting a class to keep only accurate (high probability) boxes
    * Intersection over Union (IoU) thresholding to eliminate overlapping boxes
* Because training a YOLO model from randomly initialized weights is non-trivial and requires a large dataset as well as lot of computation, previously trained model parameters can be used.

---

## Region Proposal

This section introduces the **Region Proposal** family of algorithms, specifically the **R-CNN** lineage. These methods represent a different philosophy from the "single-pass" approach of YOLO.

### The Concept of Region Proposals

The fundamental idea is to avoid running a classifier on every possible window (which is computationally wasteful in empty areas). Instead, the model only "looks" at regions where there is likely to be an interest.

* **Selective Search:** Instead of a grid, a segmentation algorithm identifies "blobs" or regions of interest (around 2,000 per image) that might contain an object.
* **Targeted Classification:** The ConvNet only processes these 2,000 proposed regions to determine if they contain a car, pedestrian, or other objects.

### The Evolution of R-CNN

1. **R-CNN (2014):** Proposes regions using a traditional algorithm and classifies them one at a time. It also performs bounding box regression ($b_x​,b_y​,b_h​,b_w​$) to refine the initial "blob" shape.
    * *Cons:* Extremely slow.
2. **Fast R-CNN (2015):** Uses a convolutional implementation of sliding windows to speed up the classification of the proposed regions.
3. **Faster R-CNN (2015):** Replaces the slow traditional segmentation algorithm with a Region Proposal Network (RPN)—a neural network that predicts the proposals themselves.
    * *Status:* Much faster than R-CNN, but generally still slower than YOLO.

### Key Takeaways

* **The Problem with R-CNN:** Even the "Faster" version is often too slow for real-world, high-speed applications compared to the single-pass convolutional logic of YOLO.
* **Utility:** While perhaps less popular for real-time tasks, R-CNN ideas are highly influential and still appear in specialized research and high-accuracy benchmarks.

---

## Semantic Segmentation with U-Net

This section introduces **Semantic Segmentation**, a sophisticated computer vision task that goes beyond simple detection by classifying every individual pixel in an image.

Unlike object recognition (identifying "what") or object detection (drawing a bounding box), semantic segmentation aims to draw a precise outline around objects.
* **Goal:** The goal is to output a label (e.g., $0, 1, 2, 3, C$) for every single pixel in the input image.
* **Commercial Applications:**
    * **Self-Driving Cars:** Identifying the exact "drivable surface" of a road rather than just putting a box around it.
    * **Medical Imaging:** Automatically outlining organs (lungs, heart) or tumors in X-rays and MRIs to assist in surgical planning and diagnosis.

<div style='display: flex; justify-content: center'>
    <img src='images/semantic_segmentation1.png' width='750px'>
</div>

<div style='display: flex; justify-content: center'>
    <img src='images/semantic_segmentation2.png' width='750px'>
</div>

### The U-Net Architecture

Here we introduce the **U-Net** as the standard architecture for this task. It differs from standard ConvNets in its shape and flow:

* **The "Down" Path (Encoder):** This is a familiar ConvNet structure. As the image passes through, the spatial dimensions (height and width) decrease while the number of channels increases to capture high-level features.
* **The "Up" Path (Decoder):** To output a full-resolution map, the network must "blow back up" the small activations. The dimensions increase and channels decrease until they match the original image size.
* **The Transpose Convolution:** This is the key mathematical operation used in the "Up" path to increase spatial resolution (upsampling).

### Key Takeaways

* **Beyond Bounding Boxes:** Segmentation is essential when the exact shape of an object matters more than just its location (e.g., an irregular brain tumor or a winding road).
* **Output Dimensions:** The output is not a vector but a matrix (or volume) that matches the height and width of the input image.
* **Transition to Math:** To implement this, you must understand the **Transpose Convolution**, which is the inverse process of a standard convolution.

---

## Transpose Convolutions

This section explains the **Transpose Convolution**, a critical mathematical operation used in the "up-sampling" path of a **U-Net** to transform small, high-level features into a full-sized segmentation map.

### The Core Concept: Scaling Up

While a normal convolution reduces spatial dimensions (e.g., $6 \times 6 \rightarrow 4 \times 4$), a transpose convolution does the opposite—it "blows up" a small input into a larger output (e.g., $2 \times 2 \rightarrow 4 \times 4$).

<div style='display: flex; justify-content: center'>
    <img src='images/transpose_conv.png' width='750px'>
</div>

### How Transpose Convolution Works

The process is conceptually the "reverse" of a standard convolution. Instead of placing the filter on the input to get one pixel, you place a *weighted version of the filter* onto the output.

* **Step-by-Step Calculation:**
1. **Scalar Multiplication:** Take a single pixel from the input (e.g., the top-left value ).
2. **Filter Weighting:** Multiply every element in your filter by that scalar.
3. **Placement:** Place the resulting block into the output grid.
4. **Stride Movement:** Move to the next input pixel. If the stride is $s$, you move your placement in the output grid by $s$ pixels.
5. **Summing Overlaps:** Where the placed filter blocks overlap in the output grid, you sum the values together.

### Parameters of Transpose Convolution

Like regular convolutions, this operation relies on three main hyperparameters:

* **Filter Size ($f$):** Usually $3 \times 3$. These weights are learned during backpropagation.
* **Padding ($p$):** Used to control the final dimensions of the output.
* **Stride ($s$):** In transpose convolutions, the stride defines how much you "stretch" the output. A stride of 2 effectively doubles the output size.

### Summary Comparison

| Feature | Regular Convolution | Transpose Convolution |
| --- | --- | --- |
| **Input  Output** | Large $\rightarrow$ Small | Small $\rightarrow$ Large |
| **Filter Usage** | Slid over the input. | Weighted and placed on the output. |
| **Goal** | Feature extraction/compression. | Spatial reconstruction (Upsampling). |
| **Mathematics** | Dot product of filter and input patch. | Scalar-filter product summed at overlaps. |

### Key Takeaways

* **Learned Upsampling:** Unlike simple "nearest neighbor" resizing, transpose convolution allows the model to learn the best way to fill in the gaps when increasing resolution.
* **The "U" in U-Net:** This operation provides the second half of the "U" shape, allowing the network to project its abstract understanding of the image back onto the original pixel coordinates.

---

## U-Net Architecture Intuition

This section explains the **U-Net**, the most iconic architecture for semantic segmentation. Its name comes from its "U" shape, consisting of a contracting path (encoder) and an expansive path (decoder).

### The U-Net Architecture

* **The Contracting Path (Encoder):**
* Uses standard convolutions and pooling to compress the image.
    * **Goal:** Captures "What" is in the image (context).
    * **Trade-off:** As the image is compressed, high-level features are identified, but precise spatial information (exactly which pixel is which) is lost.
* **The Expansive Path (Decoder):**
* Uses **Transposed Convolutions** (Upsampling) to "blow up" the representation back to the original input size.
    * **Goal:** Recovers "Where" the objects are located at a pixel level.
* **Skip Connections:** * Directly copies activation maps from the encoder layers to the corresponding decoder layers. It merges low-level, high-resolution spatial data (like edges and textures) with high-level, low-resolution contextual data (like "this region is a cat").

<div style='display:flex; justify-content: center'>
    <img src='images/u_net_arch.png' width=800px>
</div>


Here is the technical intuition behind why this specific design works so well for segmentation:

### 1. The Bottleneck Problem

In a standard CNN, by the time we reach the middle (the bottleneck), our $256 \times 256$ image might be reduced to a $16 \times 16$ grid. At this resolution, the model knows a cat exists, but it has "forgotten" exactly where the cat's ear ends and the background begins. Without a way to look back at the original resolution, the output mask would be very "blurry" or "blob-like."

### 2. Skip Connections as "Spatial Anchors"

* The decoder receives a "blurry" suggestion from the previous layer: *"Hey, I think there's a cat roughly in this area."* 
* The skip connection provides the "sharp" evidence: *"Here are the exact edges I found in step 1."*
* The model concatenates these two, allowing it to draw a razor-sharp boundary around the object.

### 3. Symmetry and Concatenation

Unlike ResNets, where skip connections are added ($x + f(x)$), U-Net skip connections are concatenated (joined along the channel axis).

* If the encoder had 64 filters and the upsampled decoder has 64 filters, the concatenated layer has 128 filters.
* This gives the model double the information to process before the next convolution.

---

## U-Net Architecture

This section provides a detailed walkthrough of the **U-Net** architecture, explaining how it transitions from an image to a pixel-level classification map.


<div style='display:flex; justify-content: center'>
    <img src='images/unet.png' width=800px>
</div>


* **Geometric Layout:** The model is named "U-Net" because its architectural diagram forms a U-shape, consisting of a downward "contracting" path and an upward "expansive" path.
* **The Contracting Path (Downsampling or Encoder):**
    * Uses standard Convolutions + ReLU activations (blue arrows).
    * Uses Max Pooling (red arrows) to reduce spatial dimensions (height/width) while increasing the number of channels (making the representation "thinner but deeper").

    <div style='display:flex; justify-content: center'>
    <img src='images/encoder.png' width=600px>
    </div>

* **The Expansive Path (Upsampling or Decoder):**
    * Uses Transposed Convolutions (green arrows) to increase spatial dimensions back toward the original image size.

    <div style='display:flex; justify-content: center'>
    <img src='images/decoder.png' width=600px>
    </div>

* **Skip Connections (The Bridge):**
    * Represented by grey arrows, these copy and concatenate activations from the contracting path directly to the expansive path.
    * This combines high-resolution spatial details from early layers with high-level contextual information from deeper layers.
* **The Final Mapping ($1 \times 1$ Convolution):**
    * The last operation (magenta arrow) is a $1 \times 1$ convolution.
    * It reduces the deep feature map into a final volume with dimensions: $H \times W \times num_classes$.
* **The Output Layer:**
    * Each pixel in the output is a vector of size $N$ (number of classes).
    * An Argmax operation is applied to each pixel's vector to determine its final class (e.g., "Car," "Pedestrian," or "Background").


>**Note:** While originally developed for biomedical image segmentation (e.g., cell tracking), U-Net is now a foundational tool for nearly all modern semantic segmentation tasks, from autonomous driving to satellite imagery.